# Evidently

Evidently is an open-source Python package for evaluating, testing, and monitoring the performance of ML models from validation to production.

Evidently has three components: Reports, Tests, and Monitors (in development). These interfaces cover alternative usage scenarios: from visual analysis to automated pipeline testing and real-time monitoring.

You need to provide the data, choose what to evaluate, and the output format. Evidently has a simple, declarative API and a library of metrics, tests, and visualizations to choose from.
You can integrate Evidently into various ML stacks as a monitoring or evaluation component.

First let's have a look at the reports. For this we will load the model from the previous notebook again.

In [ ]:
import os
from dotenv import load_dotenv

import pandas as pd

import mlflow
from evidently import ColumnMapping
from evidently.report import Report
from evidently.metrics import ColumnDriftMetric, DatasetDriftMetric, DatasetMissingValuesMetric
from evidently.test_suite import TestSuite
from evidently.test_preset import DataDriftTestPreset


from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
load_dotenv()

Set the environment variables:

In [ ]:
MLFLOW_TRACKING_URI=os.getenv("MLFLOW_TRACKING_URI")
# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
# Setup the MLflow experiment 
mlflow.set_experiment("green-taxi-monitoring")


SA_KEY=os.getenv("SA_KEY")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SA_KEY

First we will load the model from the MLFlow server. We will use the `run_id` that we can find in the MLFlow UI. 

In [ ]:
def load_model(run_id, MLFLOW_TRACKING_URI):
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    logged_model = f'runs:/{run_id}/model'
    # Load model as a PyFuncModel.
    loaded_model = mlflow.pyfunc.load_model(logged_model)
    return loaded_model

In [ ]:
run_id = "<your run id>"

loaded_model = load_model(run_id, MLFLOW_TRACKING_URI)


And prepare the data as we did in the first notebook:

In [ ]:
# Load the data
year = 2021
month = 1
color = "green"

df = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
cat_features = ["PULocationID", "DOLocationID"]
num_features = ["trip_distance", "passenger_count", "fare_amount", "total_amount"]
target = 'duration'

In [ ]:
# calculate the trip duration in minutes and drop trips that are less than 1 minute and more than 2 hours
def calculate_trip_duration_in_minutes(df):
    df["duration"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[(df['passenger_count'] > 0) & (df['passenger_count'] < 8)]
    df = df[cat_features+ num_features + [target]]
    return df

In [ ]:
df_processed = calculate_trip_duration_in_minutes(df)

In [ ]:
y=df_processed["duration"]
X=df_processed.drop(columns=["duration"])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

We will make the predictions on the train first:

In [ ]:
y_pred_train = loaded_model.predict(X_train)

train_data = X_train.copy()
train_data["duration"] = y_train
train_data["prediction"] = y_pred_train

Now on the train set:

In [ ]:
y_pred_test = loaded_model.predict(X_test)

test_data = X_test.copy()
test_data["duration"] = y_test
test_data["prediction"] = y_pred_test

And calculate the RMSE:

In [ ]:
print(mean_absolute_error(train_data.duration, train_data.prediction))
print(mean_absolute_error(test_data.duration, test_data.prediction))

Finally, we will create the Evidently report, the first step is to create a `ColumnMapping` object. This object will be used to map the columns from the data to the columns in the report.

In [ ]:
column_mapping = ColumnMapping(
    target=target,
    prediction='prediction',
    numerical_features=num_features,
    categorical_features=cat_features
)

And then we can create the report we will include the `ColumnDriftMetric`, the `DatasetDriftMetric` and the `DatasetMissingValuesMetric` in the report to detect Model Drift and Data Drift:

In [ ]:
report = Report(metrics=[
    ColumnDriftMetric(column_name='prediction'),
    DatasetDriftMetric(),
    DatasetMissingValuesMetric()
]
)

Now we have to run the report with our data, we will use the train data for the reference and the test data for the production:

In [ ]:
report.run(reference_data=train_data, current_data=test_data, column_mapping=column_mapping)

And in Jupyter Notebook we can use the `show` function to show the report:

In [ ]:
report.show(mode='inline')

Or we can save it as a dictionary and get the metrics:

In [ ]:
result = report.as_dict()

In [ ]:
#prediction drift
result['metrics'][0]['result']['drift_score']

In [ ]:
#number of drifted columns
result['metrics'][1]['result']['number_of_drifted_columns']

In [ ]:
#share of missing values
result['metrics'][2]['result']['current']['share_of_missing_values']

So we don't have any missing values and no data drifts. So we rather have drifts for the predictions nor for the data. So we did a good job for our train test split.

But what if we compare this data to a more recent one:

In [ ]:
# Load the data
year = 2022
month = 2
color = "green"

# Download the data
if not os.path.exists(f"./data/{color}_tripdata_{year}-{month:02d}.parquet"):
    os.system(f"wget -P ./data https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet")

df_21_2 = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

Preprocess the data:

In [ ]:
df_processed_21_2 = calculate_trip_duration_in_minutes(df_21_2)

Predict on the new data:

In [ ]:
y_pred_21_2 = loaded_model.predict(df_processed_21_2[cat_features+num_features])

In [ ]:
df_processed_21_2["prediction"] = y_pred_21_2

And create the report again:

In [ ]:
result = report.run(reference_data=train_data, current_data=df_processed_21_2, column_mapping=column_mapping)

In [ ]:
report.show(mode='inline')

Here we see that we don't have a drift in the prediction, but we have a worse value for the Wasserstein distance. But we have a drift in the data. We can investigate this further with the `Test Suit` of Evidently.

In [ ]:
test_suite = TestSuite(tests=[DataDriftTestPreset()])
test_suite.run(reference_data=train_data, current_data=df_processed_21_2, column_mapping=column_mapping)

In [ ]:
test_suite.show(mode='inline')

This way we will get a per column report if we have a drift or not. This integration into Jupyter Notebook is very useful for data scientists to get a quick overview of the model performance and if the train test split was well conducted.

But of course this is not how you use it in production.